In [ ]:


import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
import sys
sys.path.append(str(Path.home() / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen' / 'config'))
import plot as pt

PATH_PLOTS = Path(r"I:\Projects\Josh\RHNA\Data")
print('EXPORT Location: ' + str(PATH_PLOTS))
pd.options.display.float_format = '{:.2f}'.format

PATH_SERVER = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")
print('EXPORT Location: ' + str(PATH_SERVER))



EXPORT=True


In [ ]:

indicator = 'RHND_check_1'
plot_name = 'overcrowding'
geography = 'MPO'

df_nat = pd.read_excel(PATH_PLOTS / f"{indicator} National ACS5.xlsx", sheet_name='National')
df_mpo = pd.read_excel(PATH_PLOTS / f"{indicator} {geography} ACS5.xlsx", sheet_name=geography)

df_nat = df_nat[['NAME', 'Year', 'Overcrowding', 'Households', 'Percent']]
df_mpo = df_mpo[['NAME', 'Year', 'Overcrowding', 'Households', 'Percent']]
df_mpo['NAME'] = 'SACOG'

df_plot = pd.concat([df_nat, df_mpo])
df_plot = df_plot[df_plot['Overcrowding']=='More than one occupant per room'].reset_index(drop=True)
df_plot['Percent'] = round(df_plot['Percent']*100, 1)
display(df_plot.head())

fig = px.line(df_plot, x='Year', y='Percent', color='NAME', color_discrete_map=pt.colors_peers, markers=True)
title = '<b>Percent of Households with Overcrowding (More Than 1 Occupant Per Room)</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(ticksuffix='%')
fig.update_traces(hovertemplate='%{y}%')
fig.update_layout(legend={'traceorder': 'reversed'})


pt.plot_agol(fig, EXPORT, title, indicator, plot_name)


In [ ]:

indicator = 'Housing_1'
plot_name = 'burden'
geography = 'MPO'

df_1 = pd.read_excel(PATH_PLOTS  / f"{indicator} {geography} ACS1.xlsx", sheet_name=geography)
df_5 = pd.read_excel(PATH_PLOTS  / f"{indicator} {geography} ACS5.xlsx", sheet_name=geography)

df_1 = df_1[['MPO', 'Year', 'Cost Burden', 'Households']]
df_5 = df_5[['MPO', 'Year', 'Cost Burden', 'Households']]

df_1['Source'] = 'ACS1'
df_5['Source'] = 'ACS5'

df_plot = pd.concat([df_1, df_5])


df_plot = df_plot.pivot_table(index=['MPO', 'Year', 'Source'], columns='Cost Burden', values='Households').reset_index()
df_plot[ 'Owner Cost Burden (%)'] = df_plot[ 'Owned units over 30 pct'] / df_plot[ 'Owned units']
df_plot['Renter Cost Burden (%)'] = df_plot['Rented units over 30 pct'] / df_plot['Rented units']

df_plot = df_plot[['Year', 'Source', 'Owner Cost Burden (%)', 'Renter Cost Burden (%)']]
df_plot = df_plot.melt(id_vars=['Year', 'Source'], var_name='Cost Burden', value_name='Percent')
df_plot = df_plot[df_plot['Year']==2024]
df_plot['Percent'] = round(df_plot['Percent']*100, 1)

display(df_plot.head())

fig = px.bar(df_plot, x='Cost Burden', y='Percent', color='Source', barmode='group', color_discrete_map=pt.colors_source)
title = '<b>Cost Burden Households by Estimate (Over 30% of Income), 2024</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(dtick=25, range=[0, 102], ticksuffix='%')
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})


indicator='RHND_check_2'
pt.plot_agol(fig, EXPORT, title, indicator, plot_name)


In [ ]:


indicator = 'Cost_6'
plot_name = 'burden_all'
geography = 'MPO'

df_1 = pd.read_excel(PATH_SERVER  / f"{indicator} {geography} PUMS1.xlsx", sheet_name=geography)
df_5 = pd.read_excel(PATH_SERVER  / f"{indicator} {geography} PUMS5.xlsx", sheet_name=geography)


df_1['Source'] = 'PUMS1'
df_5['Source'] = 'PUMS5'

df_plot = pd.concat([df_1, df_5])

df_plot = df_plot[
    (df_plot['RAC1P']=='All') &
    (df_plot['Housing Type'].isin(['Owner', 'Renter'])) &
    (df_plot['Housing Burden'].isin(['Cost burden <=30%', 'Cost burden >30% to <=50%', 'Cost burden >50%']))
].reset_index(drop=True)

df_plot = df_plot[['Source', 'Year', 'Housing Type', 'Housing Burden', 'Households', 'Percent']]
df_plot['Percent'] = round(df_plot['Percent']*100, 1)


fig = px.line(df_plot, x='Year', y='Percent', color='Housing Burden', color_discrete_map=pt.colors_burden2, line_dash = 'Source', facet_col='Housing Type', markers=True)
title = '<b>Percent of Households by Cost Burden (% of Income Spent on Housing)</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(ticksuffix='%')
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})

indicator='RHND_check_3'
pt.plot_agol(fig, EXPORT, title, indicator, plot_name)




In [ ]:


indicator = 'Cost_6'
plot_name = 'burden'
geography = 'MPO'

df_1 = pd.read_excel(PATH_SERVER  / f"{indicator} {geography} PUMS1.xlsx", sheet_name=geography)
df_5 = pd.read_excel(PATH_SERVER  / f"{indicator} {geography} PUMS5.xlsx", sheet_name=geography)


df_1['Source'] = 'PUMS1'
df_5['Source'] = 'PUMS5'

df_plot = pd.concat([df_1, df_5])

df_plot = df_plot[
    (df_plot['RAC1P']=='All') &
    (df_plot['Housing Type'].isin(['Owners and Renters'])) &
    (df_plot['Housing Burden'].isin(['Cost burden <=30%', 'Cost burden >30% to <=50%', 'Cost burden >50%']))
].reset_index(drop=True)

df_plot.loc[df_plot['Housing Burden'].isin(['Cost burden >30% to <=50%', 'Cost burden >50%']), 'Housing Burden'] = 'Cost burden >30%'
df_plot = df_plot[['Source', 'Year', 'Housing Type', 'Housing Burden', 'Households', 'Percent']]
df_plot = df_plot.groupby(['Source', 'Year', 'Housing Burden'], as_index=False).agg(Households=('Households','sum'))
df_plot['Percent'] = df_plot['Households'] / df_plot.groupby(['Source', 'Year'])['Households'].transform('sum')

df_plot['Percent'] = round(df_plot['Percent']*100, 1)


fig = px.line(df_plot, x='Year', y='Percent', color='Housing Burden', color_discrete_map=pt.colors_burden2, line_dash = 'Source', markers=True)
title = '<b>Owners and Renters - Percent of Households by Cost Burden (% of Income Spent on Housing)</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(ticksuffix='%')
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})

indicator='RHND_check_3'
pt.plot_agol(fig, EXPORT, title, indicator, plot_name)


